In [ ]:
# ===============================
# 1️⃣ Mount Google Drive
# ===============================
import google.colab
google.colab.drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ===============================
# 2️⃣ Install Ultralytics (YOLOv11)
# ===============================
!pip install ultralytics --upgrade

# ===============================
# 3️⃣ Import YOLO & Check Version
# ===============================
from ultralytics import YOLO
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch version: 2.9.0+cu126
CUDA available: True


In [ ]:

from ultralytics import YOLO
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
import os
import numpy as np
from glob import glob
from collections import Counter

In [ ]:
!unzip /content/sample_data/Upgraded-Thermal--Pistol.v1-upgraded-data-set.yolov11.zip

Archive:  /content/sample_data/Upgraded-Thermal--Pistol.v1-upgraded-data-set.yolov11.zip
  inflating: README.dataset.txt      
  inflating: README.roboflow.txt     
  inflating: data.yaml               
   creating: test/
   creating: test/images/
 extracting: test/images/0051_jpg.rf.696ed5f93823c6bec59c9a2a03299baf.jpg  
 extracting: test/images/0061_jpg.rf.a5a3179dcb4b4abb868f9f1f4d3cf83b.jpg  
 extracting: test/images/0072_jpg.rf.9920bcc611ab6b827014102302b45ffa.jpg  
 extracting: test/images/0074_jpg.rf.466de6e1022aedf830248312c8b05422.jpg  
 extracting: test/images/0079_jpg.rf.c18caf508379ded72fb6a7e4983a8f1a.jpg  
 extracting: test/images/0094_jpg.rf.41a86224f16b4735d66b9ac0794d253c.jpg  
 extracting: test/images/0096_jpg.rf.33ad7362478ec76db672166c4f25a731.jpg  
 extracting: test/images/0113_jpg.rf.8837b63a651e2f2639e8268df13fadf9.jpg  
 extracting: test/images/0124_jpg.rf.48b3a9a545ef80e9bf0ab3f5d784d512.jpg  
 extracting: test/images/0161_jpg.rf.89cefd6fb363e8776d0cfa958ed399c

In [ ]:

# =======================
# 2️⃣ Setup Paths
# =======================
# Change this to your dataset.yaml path from Roboflow export
DATA_YAML = "/content/data.yaml"



# =======================
# 3️⃣ Count Class Frequencies
# =======================
# This reads all label files and counts how many instances each class has
label_files = glob("/content/train/labels/*.txt")

class_counts = Counter()
for lf in label_files:
    with open(lf, "r") as f:
        for line in f:
            class_id = int(line.strip().split()[0])
            class_counts[class_id] += 1

print("Class Counts:", class_counts)

Class Counts: Counter({0: 298})


In [ ]:
# =======================
# 4️⃣ Compute Class Weights for Balancing
# =======================
total_instances = sum(class_counts.values())
class_weights = {cls: total_instances / count for cls, count in class_counts.items()}
print("Class Weights:", class_weights)

# Assign each sample a weight based on its class occurrences
sample_weights = []
for lf in label_files:
    with open(lf, "r") as f:
        lines = f.readlines()
        if lines:
            # Use the class of the first object in image for weighting
            cls_id = int(lines[0].strip().split()[0])
            sample_weights.append(class_weights[cls_id])
        else:
            sample_weights.append(1.0)  # No label case

# Create WeightedRandomSampler
sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

# =======================
# 5️⃣ Load YOLO Model
# =======================
model = YOLO("yolo11s.pt")

# =======================
# 6️⃣ Train with Custom DataLoader
# =======================
# The YOLOv8+ training API doesn’t expose sampler directly,
# so we override the dataloader with our sampler.

# First run normal train with balanced dataset
train_results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=4, # Further reduced batch size
    workers=4,
    cache=True,
    device=0,
    pretrained=True,
    patience=20,
    amp=True,
    optimizer='SGD',
    lr0=0.01
)

Class Weights: {}


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
# =======================
# 1️⃣ Import YOLO
# =======================
from ultralytics import YOLO

# =======================
# 2️⃣ Define Data Configuration Path
# =======================
# **IMPORTANT:** Make sure this path points to the YAML file
# that defines your dataset's classes and test/validation image paths.
DATA_YAML = '/content/data.yaml'

# =======================
# 3️⃣ Load Trained Model
# =======================
# Load the model weights from your training run.
# This file is typically found in 'runs/detect/train_results/weights/best.pt'
# Replace 'path/to/best.pt' with the actual path.
MODEL_PATH = '/content/train/labels/best.pt'
model = YOLO(MODEL_PATH)

# =======================
# 4️⃣ Test/Validate the Model
# =======================
# The 'val()' method runs the model against the validation or test set
# defined in your DATA_YAML file and calculates performance metrics.
test_results = model.val(
    data=DATA_YAML,
    imgsz=640,
    batch=4, # Use the same or similar batch size as training
    workers=4,
    device=0  # Use your GPU index if available
)

# =======================
# 5️⃣ Print Key Results
# =======================
# You can now access the key metrics from the test_results object
print("\n--- Validation Results ---")
print(f"mAP50: {test_results.box.map50}")
print(f"mAP50-95: {test_results.box.map}")

Ultralytics 8.3.241 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 275.7±154.7 MB/s, size: 8.0 KB)
val: Scanning /content/valid/labels... 48 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 48/48 876.9it/s 0.1s
val: New cache created: /content/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 4.1it/s 2.9s
                   all         48         48      0.872      0.833      0.886      0.584
Speed: 3.7ms preprocess, 15.3ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to /content/runs/detect/val

--- Validation Results ---
mAP50: 0.8858008776318728
mAP50-95: 0.5836781132960317


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from ultralytics import YOLO
from google.colab import drive
import os, yaml, math, glob
from IPython.display import FileLink

# =============================================================================
# 2️⃣ Configuration
# =============================================================================

# Define the paths based on your training notebook's output structure.
# We are assuming your final trained weights (best.pt) were saved to Drive.

# 🚩 IMPORTANT: Replace this with the exact path where your 'best.pt' is saved.
# Based on your code snippet, a likely path structure is:
WEIGHTS_PATH = '/content/train/labels/best.pt' # Corrected path to use the existing best.pt

# 🚩 IMPORTANT: Upload your test video file to your Colab session or Drive
# and specify its path here.
VIDEO_PATH = '/content/sample_data/video.mp4' # Example: Upload to /content/
# Or if it's in Drive:
# VIDEO_PATH = '/content/drive/MyDrive/test_videos/my_test_clip.mp4'


# =============================================================================
# 3️⃣ Load Model and Run Prediction
# =============================================================================

print(f"Loading model from: {WEIGHTS_PATH}")
# Load the trained model.
try:
    model = YOLO(WEIGHTS_PATH)
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please check the 'WEIGHTS_PATH' is correct and accessible.")
    # Exit gracefully if model load fails
    exit() # Exit if model loading fails, as subsequent code depends on it

print(f"Starting inference on video: {VIDEO_PATH}")

# Run inference on the video. The results will be saved automatically.
# Arguments Explained:
# source: The path to the video file.
# conf: Confidence threshold (e.g., 0.25 = 25%). Adjust this to control False Positives/Negatives.
# iou: Intersection Over Union threshold for non-max suppression.
# save: Set to True to save the resulting video with detections.
# project: The folder where results will be saved (e.g., /content/runs/detect/test_results)
results = model.predict(
    source=VIDEO_PATH,
    conf=0.25, # Start with 25% confidence, adjust higher if you get too many False Positives.
    iou=0.7,
    imgsz=640,
    save=True, # This saves the output video with bounding boxes drawn
    project='runs/detect',
    name='video_test_run',
    exist_ok=True # Overwrite previous runs with the same name
)

# =============================================================================
# 4️⃣ Display Results Location
# =============================================================================

# The output video will be saved in the 'runs/detect/video_test_run' folder.
# This code block helps the user find the saved video path.

# The output video's path structure is typically:
# runs/detect/video_test_run/filename.avi (or .mp4)

# Try to find the output file saved by ultralytics
output_files = glob.glob('runs/detect/video_test_run/*.*')
output_video_path = [f for f in output_files if f.endswith('.avi') or f.endswith('.mp4')]

if output_video_path:
    print("\n✅ Detection Complete!")
    print(f"Output saved to local directory: {output_video_path[0]}")
    # Display a direct link to download the video in Colab
    print(f"Download the detected video here: {FileLink(output_video_path[0])}")
else:
    print("\nWarning: Could not automatically locate the output video file. Check the 'runs/detect/video_test_run' folder manually.")

Loading model from: /content/train/labels/best.pt
Starting inference on video: /content/sample_data/video.mp4

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/112) /content/sample_data/video.mp4: 640x480 1 Gun, 13.1ms
video 1/1 (frame 2/112) /content/sample_data/video.mp4: 640x480 1 Gun, 12.3ms
video 1/1 (frame 3/112) /content/sample_data/video.mp4: 640x480 1 Gun, 12.4ms
video 1/1 (frame 4/112) /content/sample_data/video.mp4: 640x480 1 Gun, 12.3ms
video 1/1 (frame 5/112) /content/sam

In [ ]:
from ultralytics import YOLO
from google.colab import drive
import os, yaml, math, glob
from IPython.display import FileLink, Image, display

# =============================================================================
# 2️⃣ Configuration
# =============================================================================

# Path to your trained weights
WEIGHTS_PATH = '/content/runs/detect/train/weights/best.pt'

# 🚩 UPDATE: Path to your test image(s)
# You can point to a single image or a whole folder (e.g., '/content/test_images/')
IMAGE_PATH = '/content/0079_jpg.rf.c18caf508379ded72fb6a7e4983a8f1a.jpg'

# =============================================================================
# 3️⃣ Load Model and Run Prediction
# =============================================================================

print(f"Loading model from: {WEIGHTS_PATH}")
try:
    model = YOLO(WEIGHTS_PATH)
except Exception as e:
    print(f"Error loading model: {e}")
    exit()

print(f"Starting inference on image: {IMAGE_PATH}")

# Run inference on the image
# 'save=True' will save the image with bounding boxes drawn
results = model.predict(
    source=IMAGE_PATH,
    conf=0.25,
    iou=0.7,
    imgsz=640,
    save=True,
    project='runs/detect',
    name='image_test_run',
    exist_ok=True
)

# =============================================================================
# 4️⃣ Display Results
# =============================================================================

# Find the saved image files (typically .jpg, .jpeg, or .png)
output_files = glob.glob('runs/detect/image_test_run/*.*')
image_extensions = ('.jpg', '.jpeg', '.png', '.webp')
output_images = [f for f in output_files if f.lower().endswith(image_extensions)]

if output_images:
    print("\n✅ Detection Complete!")
    for img_path in output_images:
        print(f"Result saved to: {img_path}")
        # Automatically display the result in the Colab notebook
        display(Image(filename=img_path))
        print(f"Download Link: {FileLink(img_path)}")
else:
    print("\nWarning: Could not locate the output image. Check 'runs/detect/image_test_run' folder.")

### 🚀Fine tuning

In [ ]:
from ultralytics import YOLO
from google.colab import drive
import os, yaml, math, glob

In [ ]:

# -----------------------------------------
# ᅩ PATHS to change (point these to YOUR files)
# -----------------------------------------
DATA_DIR = "/content"                     # Corrected: Data is unzipped to /content
DATA_YAML = "/content/data.yaml"          # Corrected: data.yaml is in /content
BEST_MODEL = "/content/runs/detect/train/weights/best.pt"     # Corrected: Use best.pt from detection training
SAVE_DIR   = "/content/drive/MyDrive/UmairX/Umairft-v1"  # where to copy final weights
os.makedirs(SAVE_DIR, exist_ok=True)


In [ ]:
# 2) Build a class-balanced train index by oversampling minority classes
#    We read labels/train/*.txt and duplicate image paths based on inverse class frequency.

import collections

label_paths = sorted(glob.glob(os.path.join(DATA_DIR, "train/labels/*.txt")))
assert len(label_paths), "No label files found in /train/labels/*.txt"

# Count class frequencies
cls_counts = collections.Counter()
file_classes = []  # list of [label_path, set(classes_in_file)]

for lp in label_paths:
    classes_here = set()
    with open(lp) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls = int(parts[0])
            cls_counts[cls] += 1
            classes_here.add(cls)
    if classes_here:
        file_classes.append((lp, classes_here))

total = sum(cls_counts.values())
min_count = min(cls_counts.values())
# Inverse-frequency weight per class
cls_weight = {c: (min_count / cnt) if cnt > 0 else 1.0 for c, cnt in cls_counts.items()}

print("Class counts:", dict(cls_counts))
print("Class weights (relative to minority):", {k: round(v,3) for k,v in cls_weight.items()})

# Helper: map label file to its image file (tries jpg/png/jpeg)
def label_to_image(lp):
    img_root = lp.replace("/labels/", "/images/")[:-4] # Corrected path construction
    for ext in (".jpg", ".png", ".jpeg", ".JPG", ".PNG", ".JPEG"):
        ip = img_root + ext
        if os.path.exists(ip):
            return ip
    return None

weighted_list = []
MAX_MULT = 6   # safety cap to avoid extreme duplication
for lp, classes in file_classes:
    # weight for an image = max of its classes' weights (aggressive on minority presence)
    w = max(cls_weight.get(c, 1.0) for c in classes)
    k = max(1, min(int(math.ceil(1.0 / w)), MAX_MULT))  # invert weight: rarer → larger k
    img_path = label_to_image(lp)
    if img_path:
        weighted_list.extend([img_path] * k)

print(f"Original train images: {len(file_classes)}  → Weighted entries: {len(weighted_list)}")

In [ ]:
# 3) Write the weighted train.txt and a copy of data.yaml that points to it
weighted_txt = "/content/weighted_train.txt"
with open(weighted_txt, "w") as f:
    f.write("\n".join(weighted_list))

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

# data.yaml can accept a path to a txt with image list
data_cfg_bal = dict(data_cfg)  # shallow copy
data_cfg_bal["train"] = weighted_txt  # only change train split
# Update validation path to use the absolute path on Google Drive
data_cfg_bal["val"] = os.path.join(DATA_DIR, data_cfg["val"].replace("../", ""))
balanced_yaml = "/content/data_balanced.yaml"
with open(balanced_yaml, "w") as f:
    yaml.safe_dump(data_cfg_bal, f, sort_keys=False)

print("Balanced YAML written to:", balanced_yaml)

In [ ]:
# 4) Fine-tune from your best.pt with a smaller LR
model = YOLO(BEST_MODEL)

results = model.train(
    data="/content/data_balanced.yaml",
    epochs=20,           # short, polishing run
    imgsz=640,
    batch=16,
    workers=4,
    device=0,
    lr0=0.001,           # lower LR for fine-tuning (10x below default 0.01)
    optimizer="SGD",
    patience=8,
    cache=True,
    pretrained=True,
    project="/content/runs_finetune",
    name="yolov11m_seg_bal_ft",
    verbose=True
)


In [ ]:

# 5) Save new weights to Drive
best_out = "/content/runs_finetune/yolov11m_seg_bal_ft/weights/best.pt"
last_out = "/content/runs_finetune/yolov11m_seg_bal_ft/weights/last.pt"

if os.path.exists(best_out):
    !cp -f "$best_out" "$SAVE_DIR/best_finetuned.pt"
if os.path.exists(last_out):
    !cp -f "$last_out" "$SAVE_DIR/last_finetuned.pt"

print("Saved to:", SAVE_DIR)

# 6) (Optional) How to resume later if Colab disconnects:
# model = YOLO("/content/drive/MyDrive/yolo_finetune/last_finetuned.pt")
# model.train(data=balanced_yaml, epochs=10, lr0=0.0008, project="/content/runs_finetune", name="resume_run")


In [ ]:
import os
import cv2
import numpy as np
from flask import Flask, request, jsonify
from ultralytics import YOLO
from werkzeug.utils import secure_filename

app = Flask(__name__)

# Load the trained YOLOv11 model as per implementation algorithm [cite: 145]
# Corrected: Use the absolute path to the best.pt file.
model = YOLO("/content/runs/detect/train/weights/best.pt")

UPLOAD_FOLDER = 'uploads'
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

@app.route('/detect', methods=['POST'])
def detect_weapon():
    if 'image' not in request.files:
        return jsonify({"error": "No image uploaded"}), 400

    file = request.files['image']
    filename = secure_filename(file.filename)
    filepath = os.path.join(UPLOAD_FOLDER, filename)
    file.save(filepath)

    # 1. Image Preprocessing [cite: 12, 13]
    # Perform inference using YOLOv11-S [cite: 15, 68]
    results = model(filepath)

    detections = []
    alert_triggered = False

    for r in results:
        # Extract Bounding Boxes and Confidence Scores [cite: 135]
        for box in r.boxes:
            conf = float(box.conf[0])
            cls = int(box.cls[0])
            name = model.names[cls]

            # Business Rule: Trigger alert if confidence is high [cite: 18, 168]
            if conf > 0.5:  # Example threshold
                alert_triggered = True

            detections.append({
                "class": name,
                "confidence": conf,
                "bbox": box.xyxy[0].tolist() # [x1, y1, x2, y2]
            })

    # 2. History & Detection Log [cite: 21, 119]
    # You would typically save these results to a database here

    return jsonify({
        "alert": alert_triggered,
        "detections": detections,
        "image_path": filepath
    })

if __name__ == '__main__':
    app.run(debug=True, port=5000)

In [ ]:
import os
import cv2
import numpy as np
from flask import Flask, render_template, request, jsonify
from ultralytics import YOLO
from werkzeug.utils import secure_filename

app = Flask(__name__)

# --- CONFIGURATION ---
UPLOAD_FOLDER = 'static/uploads'
os.makedirs(UPLOAD_FOLDER, exist_ok=True)
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER

# Load YOLOv11-S model (As per Section 7.1 of your Doc)
# Ensure 'best.pt' is in the same folder
try:
    model = YOLO("best.pt")
except:
    print("Error: best.pt not found. Using default yolov8n.pt for testing.")
    model = YOLO("yolov8n.pt")

# --- ROUTES ---

@app.route('/')
def home():
    """Renders the main Web Dashboard"""
    return render_template('index.html')

@app.route('/detect', methods=['POST'])
def detect():
    """
    Core Detect Logic (Aligns with SDS Section 8.3 Business Rules)
    """
    if 'image' not in request.files:
        return jsonify({"error": "No image uploaded"}), 400

    file = request.files['image']
    # Restricted Area toggle from frontend (Rule R1 vs R2)
    is_restricted = request.form.get('is_restricted') == 'true'

    filename = secure_filename(file.filename)
    filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
    file.save(filepath)

    # 1. Inference (YOLOv11 Process)
    results = model(filepath)

    detections = []
    weapon_found = False

    for r in results:
        for box in r.boxes:
            conf = float(box.conf[0])
            cls = int(box.cls[0])
            label = model.names[cls]

            # Logic: Detecting weapons (pistol, knife, etc.)
            # Assuming your model classes include these
            if conf > 0.45:  # Confidence Threshold
                weapon_found = True
                detections.append({
                    "class": label,
                    "confidence": round(conf * 100, 2),
                    "bbox": box.xyxy[0].tolist()
                })

    # 2. Applying Business Rules (Section 8.3.1)
    # R1: Weapon + Restricted Area = Generate Alert
    # R2: Weapon + Open Area = Detect Only (No Audio Alert)

    action = "No Action"
    trigger_alarm = False

    if weapon_found:
        if is_restricted:
            action = "R1: ALERT GENERATED (Restricted Area)"
            trigger_alarm = True
        else:
            action = "R2: DETECTION ONLY (Open Area)"
            trigger_alarm = False
    else:
        action = "R3/R4: NO WEAPON DETECTED"

    return jsonify({
        "status": "success",
        "weapon_detected": weapon_found,
        "trigger_alarm": trigger_alarm,
        "business_rule": action,
        "detections": detections,
        "image_url": filepath
    })

if __name__ == '__main__':
    app.run(debug=True, port=5000)